In [ ]:
#https://www.machinelearningplus.com/deep-learning/linear-regression-tensorflow/

#https://www.kaggle.com/jprukundo/on-taylor-series-neural-network-architecture

In [1]:
from numba import jit_module, jit, prange
import numpy as np

from multiprocessing import Pool

import tensorflow as tf

In [2]:
@jit(nopython = True)
def soma_vetor(x):
    return np.sum(x)

@jit(nopython = True, parallel = True)
def soma_vetor_parallel(x):
    return np.sum(x)

@jit(nopython = True)
def lista_soma_vetor(lista_x):
    return [np.sum(x) for x in lista_x]

In [3]:
n = 10000000

x = np.arange(n)
xt = tf.range(n)

In [4]:
%%timeit -n 100 -r 20

np.sum(x)

100 loops, best of 20: 9.84 ms per loop


In [5]:
%%timeit -n 100 -r 20

soma_vetor(x)

100 loops, best of 20: 7.6 ms per loop


In [6]:
%%timeit -n 100 -r 20

soma_vetor_parallel(x)

/usr/local/lib/python3.7/dist-packages/numba/np/ufunc/parallel.py:363: NumbaWarning: The TBB threading layer requires TBB version 2019.5 or later i.e., TBB_INTERFACE_VERSION >= 11005. Found TBB_INTERFACE_VERSION = 9107. The TBB threading layer is disabled.
  warnings.warn(problem)


100 loops, best of 20: 7.11 ms per loop


In [7]:
%%timeit -n 100 -r 20

tf.math.reduce_sum(xt)

KeyboardInterrupt: ignored

# Adaptando Funções para Tensorflow

In [8]:
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit, ShuffleSplit
from sklearn.model_selection import StratifiedKFold, KFold

In [9]:
df = pd.read_csv('sample_data/california_housing_train.csv')
display(df)

longitude  latitude  ...  median_income  median_house_value
0        -114.31     34.19  ...         1.4936             66900.0
1        -114.47     34.40  ...         1.8200             80100.0
2        -114.56     33.69  ...         1.6509             85700.0
3        -114.57     33.64  ...         3.1917             73400.0
4        -114.57     33.57  ...         1.9250             65500.0
...          ...       ...  ...            ...                 ...
16995    -124.26     40.58  ...         2.3571            111400.0
16996    -124.27     40.69  ...         2.5179             79000.0
16997    -124.30     41.84  ...         3.0313            103600.0
16998    -124.30     41.80  ...         1.9797             85800.0
16999    -124.35     40.54  ...         3.0147             94600.0

[17000 rows x 9 columns]

In [10]:
df.isna().sum()

longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
dtype: int64

In [11]:
nome_alvo = 'median_house_value'

In [12]:
frac_teste = 0.2

ss_teste = ShuffleSplit(n_splits = 1, test_size = frac_teste, random_state = 42)
split_teste = list(ss_teste.split(df.values, df[nome_alvo].values))[0] 

df_intermed = df.iloc[split_teste[0]]
df_teste = df.iloc[split_teste[1]]

In [13]:
frac_val = 0.5
num_splits = 2

ss = ShuffleSplit(n_splits = num_splits, test_size = frac_val, random_state = 42)
lista_splits = list(ss.split(df_intermed.values, df_intermed[nome_alvo].values))  

In [14]:
colunas_id = []

In [15]:
dict_Xy = {}

i = 0
for par in lista_splits:
    df_treino = df_intermed.iloc[par[0]]
    df_val = df_intermed.iloc[par[1]]
    
    dict_Xy.update({'Treino_' + str(2*i + 1): (df_treino.drop(colunas_id, axis = 1).drop(nome_alvo, axis = 1), df_treino[nome_alvo]),
               'Treino_' + str(2*i + 2): (df_val.drop(colunas_id, axis = 1).drop(nome_alvo, axis = 1), df_val[nome_alvo])})
    
    i = i + 1

In [16]:
from itertools import combinations_with_replacement, combinations
from itertools import groupby
from operator import itemgetter

from functools import reduce

In [17]:
class TaylorLaurentExpansion:

    def __init__(self, laurent = False, ordem = 2, apenas_interacoes = False, num_features = None):
        self.__laurent = laurent
        self.__apenas_interacoes = apenas_interacoes
        self.__num_features = num_features
        if(self.__apenas_interacoes):
            self.__ordem = min(ordem, self.__num_features)
        else:
            self.__ordem = ordem
        
        def cria_tupla(tupla):
            unique, count = np.unique(tupla, return_counts = True)
            return tuple((unique[i], count[i]) for i in range(0, unique.size))
        
        self.__lista_termos = []
        features = np.arange(0, self.__num_features)
        for i in range(1, self.__ordem + 1):
            if(self.__apenas_interacoes):
                comb = list(combinations(features, r = i)) #Não precisa de potencias das features
            else:
                comb = list(combinations_with_replacement(features, r = i))
            comb = [cria_tupla(v) for v in comb]
            self.__lista_termos.extend(comb)
        
        if(self.__laurent):
            if(self.__apenas_interacoes):
                def expande_laurent(tupla):
                    sinais = [[]]
                    tam = len(tupla)
                    for i in range(0, tam):
                        sinais_temp = sinais.copy()
                        for var in sinais_temp:
                            v_new1 = var.copy()
                            v_new2 = var.copy()
                            v_new1.append(1)
                            v_new2.append(-1)
                            sinais.pop(0)
                            sinais.append(v_new1)
                            sinais.append(v_new2)
                    #Remove os sinais que darão inversos multiplicativos
                    sinais_filtrados = []
                    while(len(sinais) > 1):
                        sinal = sinais[0]
                        sinais_filtrados.append(sinal)
                        sinais.remove(sinal)
                        try:
                            sinais.remove(list(-1*np.array(sinal)))
                        except:
                            pass
                    if(len(sinais) == 1):
                        sinais_filtrados.append(sinais[0])
                    sinais = sinais_filtrados
                    return [tuple((tupla[j][0], s[j]*tupla[j][1]) for j in range(0, tam)) for s in sinais]
            else:
                def expande_laurent(tupla):
                    sinais = [[]]
                    tam = len(tupla)
                    for i in range(0, tam):
                        sinais_temp = sinais.copy()
                        for var in sinais_temp:
                            v_new1 = var.copy()
                            v_new2 = var.copy()
                            v_new1.append(1)
                            v_new2.append(-1)
                            sinais.pop(0)
                            sinais.append(v_new1)
                            sinais.append(v_new2)
                    return [tuple((tupla[j][0], s[j]*tupla[j][1]) for j in range(0, tam)) for s in sinais]
            lista_aux = []
            for tupla in self.__lista_termos:
                lista_aux.extend(expande_laurent(tupla))
            self.__lista_termos = lista_aux
        
        self.__num_termos = len(self.__lista_termos)
    
    def filtro_binario(self, lista_features_binarias):
        def eh_termo_indevido(termo, feature):
            for v in termo:
                if(v[0] == feature and v[1] != 1):
                    return True
            return False
        for feature in lista_features_binarias:
            self.__lista_termos = [termo for termo in self.__lista_termos if eh_termo_indevido(termo, feature) == False]
        self.__num_termos = len(self.__lista_termos)
    
    def numero_termos_expansao(self):
        return self.__num_termos
        
    def lista_termos(self):
        return self.__lista_termos

    def calcula_termo(self, X, pos_termo):
        return calcula_termo_serie(X, self.__lista_termos[pos_termo])

In [70]:
#@tf.function(jit_compile = True)
def calcula_termo_serie_tf(X, termo):
    #return reduce(lambda x, y: tf.math.multiply(x, y), [tf.math.pow(X[:, c[0]], c[1]) for c in termo])
    return tf.reduce_prod(tf.map_fn(lambda i: tf.math.pow(X[:, termo[i][0]], termo[i][1]), tf.range(len(termo)), dtype = tf.dtypes.float64), axis = 0)

In [19]:
X = dict_Xy['Treino_1'][0].values
y = dict_Xy['Treino_1'][1].values

In [53]:
series = TaylorLaurentExpansion(laurent = True, ordem = 5, apenas_interacoes = False, num_features = X.shape[1])
termos = series.lista_termos()
termos = [tf.tuple([(c[0], float(c[1])) for c in termo]) for termo in termos]
num_termos = series.numero_termos_expansao()
print(num_termos)

13072


In [71]:
def calcula_termos(X, termos):
  return tf.map_fn(lambda i: calcula_termo_serie_tf(X, termos[i]), tf.range(num_termos), dtype = tf.dtypes.float64)

def calcula_medias(X, termos):
  return tf.math.reduce_mean(calcula_termos(X, termos), axis = 1)

def calcula_desvios(X, termos):
  return tf.math.reduce_std(calcula_termos(X, termos), axis = 1)

In [72]:
#medias = tf.math.reduce_mean(tf.convert_to_tensor([calcula_termo_serie(X, termo) for termo in termos]), axis = 1)
#desvios = tf.math.reduce_std(tf.convert_to_tensor([calcula_termo_serie(X, termo) for termo in termos]), axis = 1)
medias = calcula_medias(X, termos)
desvios = calcula_desvios(X, termos)

media_y = tf.math.reduce_mean(y)
desvio_y = tf.math.reduce_std(y)

y_norm = (y - media_y)/desvio_y

In [73]:
alpha = 0.01
num_epocas = 100

In [74]:
bias = tf.Variable(tf.math.reduce_mean(y_norm), dtype = tf.dtypes.float64)
weights = tf.Variable(tf.zeros(num_termos, dtype = tf.dtypes.float64))

In [75]:
def hipotese(X):
  #y = bias + reduce(lambda x, y: tf.math.add(x, y), [weights[i]*(calcula_termo_serie(X, termos[i]) - medias[i])/desvios[i] for i in range(num_termos)])
  y = bias + tf.reduce_sum(tf.map_fn(lambda i: weights[i]*calcula_termo_serie_tf(X, termos[i]), tf.range(num_termos), dtype = tf.dtypes.float64))
  return y

In [76]:
def funcao_custo(y_pred, y_real):
  return tf.reduce_mean(tf.square(y_pred - y_real))

In [77]:
y_pred = hipotese(X)
custo_ini = funcao_custo(y_pred, y_norm)

In [ ]:
for epoca in range(num_epocas):

  with tf.GradientTape() as tape:
    y_pred = hipotese(X)
    custo = funcao_custo(y_pred, y_norm)
    grad = tape.gradient(custo, [bias, weights])

  norma_quadrada_gradiente = tf.sqrt(tf.reduce_sum(tf.square(grad[0]) + tf.square(grad[1])))
  cte = custo_ini/norma_quadrada_gradiente
  bias.assign_sub(grad[0]*cte*alpha)
  weights.assign_sub(grad[1]*cte*alpha)

  print(f"Epoca {epoca}: R2: {1 - custo.numpy()/custo_ini}")

ERROR:tensorflow:==================================
Object was never used (type <class 'tensorflow.python.ops.tensor_array_ops.TensorArray'>):
If you want to mark it as used call its "mark_used()" method.
It was originally created here:
  File "/usr/local/lib/python3.7/dist-packages/tensorflow/python/ops/control_flow_ops.py", line 2820, in while_loop
    return result  File "/usr/local/lib/python3.7/dist-packages/tensorflow/python/ops/control_flow_ops.py", line 2768, in <lambda>
    body = lambda i, lv: (i + 1, orig_body(*lv))  File "/usr/local/lib/python3.7/dist-packages/tensorflow/python/ops/map_fn.py", line 499, in compute
[... 68 linhas omitidas na versão pública ...]
Epoca 64: R2: -6490712005.084108
Epoca 65: R2: -3.645467451495397e+42
Epoca 66: R2: -113773529987.93999
Epoca 67: R2: -3.6454674514953936e+42
Epoca 68: R2: -225039373147.13162
Epoca 69: R2: -3.6454674514953936e+42
Epoca 70: R2: -394498027321.6221
Epoca 71: R2: -3.6454674514953936e+42
